# Pixel Transformer tutorial: from raw hits to a classifier

This is the short tutorial notebook. It jumps straight into a small Transformer-like model for pixel clusters.

The goal is not to build the most complicated architecture. The goal is to understand the core idea:

> represent each hit, let hits exchange information through attention, pool the learned hit information, then classify the cluster.

You will start from a simple implementation and then have space to try improvements. The leaderboard at the end compares:

- a pretrained BDT reference,
- a pretrained/simple Transformer reference,
- any models you train in this notebook.

In [ ]:
import sys, os, json, math, time, random
from pathlib import Path

# FILL THIS PATH!
PROJECT_ROOT = Path('')
TUTORIAL_DIR = PROJECT_ROOT / '/2026-gatech/sessions/04_muon_col/tutorial'

import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from dataloader import load_pixel_data
from tutorial_utils import (
    Trainer, build_feature_matrix, classifier_summary_row,
    plot_score_distributions, plot_confusion_matrix,
    plot_roc_comparison, plot_leaderboard,
    evaluate_classifier, predict_model,
    benchmark_model, benchmark_bdt, print_benchmark,
    count_parameters, display_model_card,
)


## Configuration

In [ ]:
SIGNAL_H5 = str(PROJECT_ROOT / 'Samples_v2/signal.h5')
BIB_H5    = str(PROJECT_ROOT / 'Samples_v2/BIB.h5')

MAX_RAW_HITS = 50
TEST_FRACTION = 0.2
VAL_FRACTION  = 0.1
RANDOM_SEED   = 42
BATCH_SIZE    = 256

OUTPUT_DIR = TUTORIAL_DIR / 'simple_transformer_tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BDT_MODEL_PATH = TUTORIAL_DIR / 'bdt_baseline_outputs' / 'bdt_gradient_boosting.joblib'
PRETRAINED_TRANSFORMER_PATH = OUTPUT_DIR / 'simple_pixel_transformer_reference.pt'

# If the pretrained Transformer file is not present, the notebook can train it from scratch.
TRAIN_REFERENCES_IF_MISSING = True

for required_path in [SIGNAL_H5, BIB_H5]:
    if not Path(required_path).exists():
        raise FileNotFoundError(required_path)


In [ ]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            device = 'cuda'
        except RuntimeError:
            print('CUDA device found but not usable. Falling back to CPU.')
    return device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
device = get_device()
print('Using device:', device)

## Load raw-hit data

In [ ]:
data = load_pixel_data(
    SIGNAL_H5, BIB_H5,
    max_hits=MAX_RAW_HITS,
    batch=BATCH_SIZE,
    test_frac=TEST_FRACTION,
    val_frac=VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_loader = data['train']
val_loader   = data['val']
test_loader  = data['test']
labels       = np.asarray(data['labels']).astype(int)
idx_train    = np.asarray(data['idx_train'])
idx_val      = np.asarray(data['idx_val'])
idx_test     = np.asarray(data['idx_test'])

print(f"Loaded {len(labels):,} clusters: {labels.sum():,} signal, {(labels == 0).sum():,} BIB")
print(f"Train: {len(idx_train):,}, Val: {len(idx_val):,}, Test: {len(idx_test):,}")
print(f"Sequence length: {MAX_RAW_HITS}")
print('Input features per hit: energy, time, x, y')

## The simple Transformer

A few simplifying choices:

- one attention head,
- simple dot-product attention,
- simple masking for empty raw hits,
- sum pooling at the end,
- small MLP classifier head.

This model is easy to read, but does not succeed at providing incredible discrimination between singal and BIB.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, x, mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = attn @ v

        return x * mask


class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.att = Attention(dim)
        self.proj1 = nn.Linear(dim, dim)
        self.proj2 = nn.Linear(dim, dim)
        self.activation = nn.GELU()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.att(self.norm1(x), mask)
        x = self.activation(self.proj1(x)) * mask
        x = x + self.proj2(self.norm2(x)) * mask
        return x


class SimplePixelTransformer(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=3, num_classes=2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        # A hit is real if the energy feature is nonzero.
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        # Sum pooling preserves extrema information better than a pure mean.
        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)

## Train or load the reference simple Transformer

In [ ]:
config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}

reference_transformer = SimplePixelTransformer(**config).to(device)
display_model_card(reference_transformer, config)

if PRETRAINED_TRANSFORMER_PATH.exists():
    print('Loading pretrained Transformer:', PRETRAINED_TRANSFORMER_PATH)
    reference_transformer.load_state_dict(torch.load(PRETRAINED_TRANSFORMER_PATH, map_location=device))
elif TRAIN_REFERENCES_IF_MISSING:
    print('No pretrained Transformer found. Training reference model...')
    trainer = Trainer(
        train_dataset=train_loader,
        val_dataset=val_loader,
        model=reference_transformer,
        lr=3e-4,
        optimizer=torch.optim.Adam,
        loss_fn=nn.CrossEntropyLoss,
        device=device,
    )
    trainer.train(num_epochs=60, patience=8)
    reference_transformer = trainer.model
    torch.save(reference_transformer.state_dict(), OUTPUT_DIR / 'simple_pixel_transformer_reference.pt')
else:
    print('Using randomly initialized reference_transformer. Set TRAIN_REFERENCES_IF_MISSING=True to train.')

## Student Exercise: try to improve the model!

Below I list a few ideas on how you can make the model better. They are in order of how easy the change would be (and also, sort of, in order of how effective the change might be). Feel free to try any of these, or an idea of your own!

- **Change the model capacity.** Increase `hidden_dim` from 64 to 96 or 128. This gives each hit more learned features, but it also increases training time.
- **Change the depth.** Try 2, 3, or 4 Transformer blocks. In our tests, 2--3 layers were usually enough; more layers did not automatically help.
- **Try a different pooling choice.** The reference uses sum pooling because it preserves some occupancy information. Mean pooling is also reasonable, but it divides out the number of hits.
- **Try giving the model more hits.** We currently cap at 50 hits, by changing that in the dataloading (and re-loading the data) you can give the model more information
- **Add light dropout.** Try `dropout=0.05` or `0.10` in the attention weights or classifier head. This can help if validation performance is noisy.
- **Preprocess hit `energy` and `time` before the input layer.** A useful simple version is:
  - `energy -> log1p(energy)`, because energy can have a long positive tail;
  - `time -> asinh(time)`, because time can have large positive values but may also have small/negative values;
  - then standardize each hit feature using the mean and standard deviation computed on real training hits only.

    For example, conceptually:

    ```python
    x[:, :, 0] = torch.log1p(torch.clamp(x[:, :, 0], min=0))
    x[:, :, 1] = torch.asinh(x[:, :, 1])
    x = (x - feature_mean) / feature_std
    ```

The important distinction is that this is still using primitive hit information. It is not giving the network engineered cluster quantities like `cluster_size_y` or `incident_angle`.

- **Give the model primitive detector context in the advanced notebook.** For example, cluster position variables like `(cluster_x, cluster_y, cluster_z, cluster_r)` helped because local pixel coordinates alone do not contain all detector-position information.

You should modify the class above or copy it below into `MyPixelTransformer`.

In [ ]:
# Example model: edit this cell or copy SimplePixelTransformer above.
class MyPixelTransformer(SimplePixelTransformer):
    pass

my_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}

TRAIN_MY_MODEL = False

if TRAIN_MY_MODEL:
    my_model = MyPixelTransformer(**my_config).to(device)
    trainer = Trainer(
        train_dataset=train_loader,
        val_dataset=val_loader,
        model=my_model,
        lr=3e-4,
        optimizer=torch.optim.Adam,
        loss_fn=nn.CrossEntropyLoss,
        device=device,
    )
    trainer.train(num_epochs=60, patience=8)
    my_model = trainer.model
else:
    my_model = None

## Load the BDT reference from notebook 01

The BDT is saved. If the file is missing, run `01_bdt_baseline.ipynb` first.


In [ ]:
X_bdt_all = build_feature_matrix(data, data['feature_keys'])
X_bdt_test = X_bdt_all[idx_test]
y_test = labels[idx_test]

if not BDT_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'BDT not found at {BDT_MODEL_PATH}. Run 01_bdt_baseline.ipynb first.'
    )

print('Loading BDT reference:', BDT_MODEL_PATH)
bdt = joblib.load(BDT_MODEL_PATH)


## Visual comparison: BDT, reference Transformer, and your model

The final cells collect scores for each available model and make the some plots.


In [ ]:
def score_torch_model(model, loader, name):
    scores, y = predict_model(model, loader, device)
    row = classifier_summary_row(name, 'Transformer', y, scores)
    return row, np.asarray(y), np.asarray(scores)

summary_rows = []
score_payloads = {}
leaderboard_plot_entries = []

# 1. BDT reference from notebook 01.
bdt_scores = bdt.predict_proba(X_bdt_test)[:, 1]
bdt_row = classifier_summary_row('BDT reference', 'BDT', y_test, bdt_scores, BDT_MODEL_PATH)
summary_rows.append(bdt_row)
score_payloads['BDT reference'] = (y_test, bdt_scores)
bdt_bench = benchmark_bdt(bdt, bdt_row['test_auc'])
leaderboard_plot_entries.append(('BDT reference', bdt_bench['auc'], bdt_bench['size_bytes'], bdt_bench['fom']))

# 2. Reference Transformer from this notebook.
row, y_tf, scores_tf = score_torch_model(reference_transformer, test_loader, 'SimplePixelTransformer reference')
summary_rows.append(row)
score_payloads['SimplePixelTransformer reference'] = (y_tf, scores_tf)
tf_bench = benchmark_model(reference_transformer, test_loader, device=device, num_warmup=2, num_runs=10)
print_benchmark(tf_bench, 'SimplePixelTransformer reference')
leaderboard_plot_entries.append(('SimplePixelTransformer reference', tf_bench['auc'], tf_bench['size_fp32_bytes'], tf_bench['fom_fp32']))

# 3. Student model, if TRAIN_MY_MODEL=True above.
if my_model is not None:
    row, y_my, scores_my = score_torch_model(my_model, test_loader, 'MyPixelTransformer')
    summary_rows.append(row)
    score_payloads['MyPixelTransformer'] = (y_my, scores_my)
    my_bench = benchmark_model(my_model, test_loader, device=device, num_warmup=2, num_runs=10)
    print_benchmark(my_bench, 'MyPixelTransformer')
    leaderboard_plot_entries.append(('MyPixelTransformer', my_bench['auc'], my_bench['size_fp32_bytes'], my_bench['fom_fp32']))
else:
    print('MyPixelTransformer is not included because TRAIN_MY_MODEL=False.')

summary = pd.DataFrame(summary_rows).sort_values('test_auc', ascending=False)
summary.to_csv(OUTPUT_DIR / 'tutorial_leaderboard.csv', index=False)
print('Saved summary to', OUTPUT_DIR / 'tutorial_leaderboard.csv')


In [ ]:
# Draw the same visual diagnostics for every available model.
roc_payload = {}
for name, (y, scores) in score_payloads.items():
    roc_payload[name] = evaluate_classifier(y, scores, name)

plot_roc_comparison(roc_payload)

for name, (y, scores) in score_payloads.items():
    plot_score_distributions(y, scores, title=f'{name} score distribution')
    plot_confusion_matrix(y, scores, cut=0.5, title=f'{name} confusion matrix, cut = 0.5')

plot_leaderboard(leaderboard_plot_entries)
